# Enhanced Cleaning — Detik
Remove trailing boilerplate:
- `Simak juga...`
- `Saksikan ulasan selengkapnya hanya di...`
- `Simak video...`
- `Scroll to continue with content`

In [ ]:
import re
import pandas as pd

DATA_PATH   = "/kaggle/input/datasets/davinraffilio9/datalabeled/data_labeled"
OUTPUT_PATH = "/kaggle/working"

In [ ]:
df = pd.read_csv(f"{DATA_PATH}/detik_labeled.csv")
print(f"Loaded: {df.shape}")
df.head(3)

## EDA — Before Cleaning

In [ ]:
print("=== Label Distribution ===")
print(df["label"].value_counts())
print(f"\nAvg content length: {df['content'].str.len().mean():.0f} chars")

checks = {
    "Simak juga"                           : r"Simak juga",
    "Saksikan ulasan selengkapnya hanya di": r"Saksikan ulasan selengkapnya hanya di",
    "Simak video"                          : r"[Ss]imak\s+[Vv]ideo",
    "Scroll to continue"                   : r"[Ss]croll\s+to\s+continue",
}
for name, pat in checks.items():
    n = df["content"].str.contains(pat, na=False, regex=True).sum()
    print(f"  {name:<45}: {n} rows")

## Cleaning — Truncate at Earliest Trigger Phrase

In [ ]:
# Potong dari titik kemunculan paling awal dari semua pattern
DETIK_CUT_PATTERNS = [
    r"Simak juga[^\n]*",
    r"Saksikan ulasan selengkapnya hanya di[^\n]*",
    r"[Ss]imak\s+[Vv]ideo[^\n]*",
    r"[Ss]croll\s+to\s+continue\s+with\s+content[^\n]*",
]

def truncate_detik(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text)
    earliest = len(text)
    for pattern in DETIK_CUT_PATTERNS:
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m and m.start() < earliest:
            earliest = m.start()
    text = text[:earliest].rstrip()
    return re.sub(r"\s+", " ", text).strip()

df["content_clean"] = df["content"].apply(truncate_detik)
affected = (df["content_clean"] != df["content"].fillna("")).sum()
print(f"Done. Rows affected: {affected} / {len(df)}")

## Before / After Comparison

In [ ]:
show = [
    ("Simak juga",           r"Simak juga"),
    ("Saksikan ulasan",      r"Saksikan ulasan selengkapnya hanya di"),
]
for label, pat in show:
    mask = df["content"].str.contains(pat, case=False, na=False)
    if not mask.any():
        print(f"No examples: {label}"); continue
    row = df[mask].iloc[0]
    raw = str(row["content"])
    idx = re.search(pat, raw, re.IGNORECASE).start()
    print(f"=== BEFORE ({label}) ===")
    print(raw[max(0, idx-100):idx+200])
    print("\n=== AFTER (same region) ===")
    print(str(row["content_clean"])[max(0, idx-100):idx+50])
    print("-" * 70)

print(f"\nAvg length BEFORE: {df['content'].str.len().mean():.0f}")
print(f"Avg length AFTER : {df['content_clean'].str.len().mean():.0f}")

## Rebuild `text` = title + content_clean

In [ ]:
df["text"] = (
    df["title"].astype(str).str.strip() + ". " +
    df["content_clean"].astype(str).str.strip()
).str.strip()

df_out = df[["date", "title", "content_clean", "article_id", "text", "label"]].copy()
df_out = df_out.rename(columns={"content_clean": "content"})

print(f"Output shape : {df_out.shape}")
print(df_out["label"].value_counts())
df_out.head(3)

In [ ]:
df_out.to_csv(f"{OUTPUT_PATH}/detik_labeled.csv", index=False, encoding="utf-8")
print(f"Saved: {OUTPUT_PATH}/detik_labeled.csv  ({len(df_out)} rows)")